# `PHREEQCChemicalEquilibriumEngine` — Instantiation and Use

Three engines satisfy `ChemicalEquilibriumEngineProtocol`:
`BisectionChemicalEquilibriumEngine`, `NRChemicalEquilibriumEngine`, and
`PHREEQCChemicalEquilibriumEngine`. None is "the" default — pick the one
that matches your chemistry's shape. This notebook covers
`PHREEQCChemicalEquilibriumEngine`: a black-box wrapper around
[`phreeqpython`](https://github.com/Vitens/phreeqpython) (an **optional**
dependency — `pip install PyOMES[phreeqc]`), delegating to PHREEQC's own
thermodynamic database and ion-pair library rather than PyOMES-declared
`EquilibriumReaction` networks.

**Contrast with `BisectionChemicalEquilibriumEngine`/`NRChemicalEquilibriumEngine`:**
this is the most structurally different of the three. It has no
`from_reactions()` classmethod at all — chemistry comes from PHREEQC's
own database via a `component_map` (PyOMES id → PHREEQC element key), not
from declared `EquilibriumReaction`/`HenryEquilibrium`/`KspEquilibrium`
instances. It satisfies only the black-box tier of the protocol hierarchy
(`solve()`, `algebraic_species()`, `reset_cache()`, `reset_counters()`) —
wrap it with `NumericalGradientEquilibriumEngine` for `jacobian_dz_dy()`
rather than expecting it directly, unlike `NRChemicalEquilibriumEngine`
which supports `retain_jacobian=True` natively.

If `phreeqpython` is not installed, constructing this engine raises a
clear `ImportError` with installation instructions rather than failing
obscurely later — this notebook assumes it's already installed (it's
installed in this kernel's environment).

For a deep accuracy comparison against the other two engines, see
[`../../model_api/chemistry/speciation/06_phreeqc_benchmark.ipynb`](../../model_api/chemistry/speciation/06_phreeqc_benchmark.ipynb)
and
[`../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb`](../../model_api/chemistry/speciation/10_engine_protocol_hierarchy.ipynb).
See also [`01_bisection_engine_basics.ipynb`](01_bisection_engine_basics.ipynb)
and [`02_nr_engine_basics.ipynb`](02_nr_engine_basics.ipynb) for the other
two engines.

In [1]:
import sys
from pathlib import Path

def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

sys.path.insert(0, str(_find_repo() / "models"))

from PyOMES.chemical_equilibrium.phreeqc_engine import (
    PHREEQCChemicalEquilibriumEngine, phreeqc_to_vlsim,
)

print("Imports OK")


Imports OK


## 1  Instantiation: `component_map`, not `from_reactions()`

Construction takes a priming composition (`{vlsim_id: mmol/L}`) plus a
`component_map` translating each PyOMES component id to a PHREEQC element
key (oxidation-state notation where the element has more than one, e.g.
`"C(4)"` for carbon(IV) as CO2/carbonate, `"N(-3)"` for reduced nitrogen
as NH3/NH4+). The priming solve both validates the mapping immediately
(a bad key raises during PHREEQC's own parse, not silently) and caches a
conservative superset of species for `algebraic_species()`.

In [2]:
engine = PHREEQCChemicalEquilibriumEngine(
    {"CO2": 1.0, "Ca": 0.5},                       # priming composition, mmol/L
    component_map={"CO2": "C(4)", "Ca": "Ca"},
)

print("Engine constructed:  ", type(engine).__name__)
print("component_map:       ", engine.component_map)
print("n_solve_calls:       ", engine.n_solve_calls)


Engine constructed:   PHREEQCChemicalEquilibriumEngine
component_map:        {'CO2': 'C(4)', 'Ca': 'Ca'}
n_solve_calls:        0


## 2  `solve()` and the `totals=` dict

`solve()` uses the **same `totals=` dict convention as
`NRChemicalEquilibriumEngine`** — `{vlsim_component_id: mol_L}` — not
`BisectionChemicalEquilibriumEngine`'s named kwargs. There is no
`strong_ions=` split: every component (including strong ions like Ca) goes
through the same `component_map`/`totals` path, since PHREEQC's own
database handles the full ion-pairing network.

In [3]:
result = engine.solve(totals={"CO2": 0.010, "Ca": 0.002})

print(f"pH               = {result.pH:.4f}")
print(f"logH             = {result.logH:.4f}")
print(f"ionic_strength   = {result.ionic_strength:.4e} mol/L")
print(f"n_solve_calls    = {engine.n_solve_calls}")


pH               = 7.2865
logH             = -7.2865
ionic_strength   = 7.2648e-03 mol/L
n_solve_calls    = 1


## 3  Species name translation: `phreeqc_to_vlsim`

PHREEQC's own species names use charge-number notation (`Ca+2`, `CO3-2`);
PyOMES convention repeats the sign (`Ca++`, `CO3--`). The default
`species_map=phreeqc_to_vlsim` translates every key in `sol.species`
before it lands in `result.species_mol_L` — this is what lets a calcium
carbonate system (declared only via `component_map`, no hand-built
`EquilibriumReaction` network) come back with names consistent with the
rest of PyOMES.

In [4]:
print("Ca+2  ->", phreeqc_to_vlsim("Ca+2"))
print("CO3-2 ->", phreeqc_to_vlsim("CO3-2"))
print("HCO3- ->", phreeqc_to_vlsim("HCO3-"), " (pass-through, already singly-charged)")

print()
print("From the solve() above:")
for sp in sorted(result.species_mol_L):
    if sp in ("Ca++", "CO3--", "CaCO3", "CaHCO3+", "HCO3-", "CO2"):
        print(f"  {sp:<10} {result.species_mol_L[sp]:.6e} mol/L")


Ca+2  -> Ca++
CO3-2 -> CO3--
HCO3- -> HCO3-  (pass-through, already singly-charged)

From the solve() above:
  CO2        5.172995e-04 mol/L
  CO3--      5.723782e-06 mol/L
  Ca++       2.384355e-03 mol/L
  CaCO3      1.141119e-05 mol/L
  CaHCO3+    1.042684e-04 mol/L
  HCO3-      4.861379e-03 mol/L


## 4  Gotcha: warmstart drifts from a fresh solve

`use_warmstart=True` (the default) reuses the PHREEQC `Solution` object
across calls via `sol.change(...)`, which issues a PHREEQC `REACTION`
block — an **incremental** addition relative to whatever composition the
solution already holds — rather than resetting to an absolute total. A
fresh engine (`use_warmstart=False`) creates a brand-new `Solution` on
every call instead, always starting from the declared total.

For a single one-shot `solve()` call these agree closely. Across a
*sequence* of calls with changing totals, they can drift apart by more
than the "should agree" tolerance used in this repo's own test suite
(`abs=0.02` pH) — this is a known, currently-undocumented-elsewhere
limitation of the warmstart path, not a numerical-precision artifact.
Reach for `use_warmstart=False` (or `reset_cache()` between calls) if you
need exact agreement with a fresh solve at every step.

In [5]:
warm = PHREEQCChemicalEquilibriumEngine({"CO2": 1.0}, component_map={"CO2": "C(4)"}, use_warmstart=True)
cold = PHREEQCChemicalEquilibriumEngine({"CO2": 1.0}, component_map={"CO2": "C(4)"}, use_warmstart=False)

print(f"{'CT_CO2':>8} {'warm pH':>10} {'cold pH':>10} {'|diff|':>8}")
for ct in (0.001, 0.005, 0.010, 0.050):
    out_w = warm.solve(totals={"CO2": ct})
    out_c = cold.solve(totals={"CO2": ct})
    print(f"{ct:>8.3f} {out_w.pH:>10.4f} {out_c.pH:>10.4f} {abs(out_w.pH - out_c.pH):>8.4f}")


  CT_CO2    warm pH    cold pH   |diff|
   0.001     4.5916     4.6805   0.0889
   0.005     4.3772     4.3284   0.0487
   0.010     4.2003     4.1773   0.0231
   0.050     3.9109     3.8266   0.0844


## 5  `algebraic_species()`, `reset_cache()`, `reset_counters()`

- `algebraic_species()` — a **conservative superset** discovered at the
  priming solve, translated through `species_map`. Species with
  negligible concentration at the priming point are still included;
  they'll just have near-zero values in practice. Unlike the Bisection/NR
  engines (whose algebraic species come from the declared reaction
  graph), this set comes from whatever PHREEQC's database considers
  possible for the priming composition.
- `reset_cache()` — discards the warmstart `Solution`; the next
  `solve()` builds a fresh one (a one-shot escape from Section 4's
  drift, at the cost of losing the warmstart speed benefit for that call).
- `reset_counters()` — resets `n_solve_calls` to zero.

In [6]:
alg = engine.algebraic_species()
print("algebraic_species() sample:", sorted(alg)[:10])
print("'Ca++' in algebraic_species():", "Ca++" in alg)
print("'CO3--' in algebraic_species():", "CO3--" in alg)

print("\nn_solve_calls before reset:", engine.n_solve_calls)
engine.reset_counters()
print("n_solve_calls after reset_counters():", engine.n_solve_calls)

engine.reset_cache()
print("_sol after reset_cache():", engine._sol)


algebraic_species() sample: ['CO2', 'CO3--', 'Ca++', 'CaCO3', 'CaHCO3+', 'CaOH+', 'H+', 'H2', 'H2O', 'HCO3-']
'Ca++' in algebraic_species(): True
'CO3--' in algebraic_species(): True

n_solve_calls before reset: 1
n_solve_calls after reset_counters(): 0
_sol after reset_cache(): None


## Summary

| Topic | Key takeaway |
|---|---|
| Construction | Direct `__init__(components, component_map=...)` — **no** `from_reactions()`; chemistry comes from PHREEQC's own database, not declared `EquilibriumReaction` networks |
| Dependency | Optional (`phreeqpython`); raises a clear `ImportError` at construction if missing |
| `solve()` totals | `totals={component_id: mol_L}` dict — same shape as `NRChemicalEquilibriumEngine`, no `strong_ions=` split |
| Name translation | `species_map=phreeqc_to_vlsim` (default) converts `Ca+2`→`Ca++`, `CO3-2`→`CO3--`, etc.; pass `None` for raw PHREEQC names |
| Protocol tier | Black-box only — wrap with `NumericalGradientEquilibriumEngine` for `jacobian_dz_dy()`, unlike NR's native `retain_jacobian=True` |
| Warmstart gotcha | `use_warmstart=True` reacts incrementally (PHREEQC `REACTION` block) rather than resetting to an absolute total — drifts from a fresh solve across a sequence of calls; use `use_warmstart=False` or `reset_cache()` for exact agreement |
| `algebraic_species()` | Conservative superset from the priming solve, not derived from a declared reaction graph |